This generator works for any position (even if tablebase not available).

In [13]:
import networkx as nx
import chess
import chess.syzygy
import chess.svg
from stockfish import Stockfish

In [2]:
tablebase = chess.syzygy.open_tablebase("C:/syzygy/syzygy_345")

In [35]:
def get_node_name(board, starting_color, stockfish, decisive_threshold=2.0, which="wdl"):
    epd_i = board.epd()
    color = epd_i[-5]
    n_pieces = len(board.piece_map())

    if which != "eval":
        if n_pieces <=5: 
            wdl = tablebase.probe_wdl(board)
            if color != starting_color:
                wdl = -wdl
        else:
            stockfish.set_fen_position(board.fen())
            eval = stockfish.get_static_eval()
            if eval is None:
                w, d, l = stockfish.get_wdl_stats()
                if w > d and w > l:
                    wdl = 2
                elif d > w and d > l:
                    wdl = 0
                elif l > d and l > w:
                    wdl = -2
                else: # shouldn't happen
                    wdl = 0
            elif eval > decisive_threshold:
                wdl = 2
            elif eval < -decisive_threshold:
                wdl = -2
            else:
                wdl = 0
            if color != starting_color: # why are we doing this?!
                wdl = -wdl
    
    if which != "wdl":
        stockfish.set_fen_position(board.fen())
        eval = stockfish.get_static_eval()
        if eval is None:
            w, d, l = stockfish.get_wdl_stats()
            if w > d and w > l:
                ev = 1500
            elif d > w and d > l:
                ev = 0
            elif l > d and l > w:
                ev = -1500
            else: # shouldn't happen
                ev = 0
        else:
            ev = eval

    if which == "wdl":
        return epd_i[0 : len(epd_i) - 6] + ":" + color + ":" + str(wdl)
    elif which == "eval":
        return epd_i[0 : len(epd_i) - 6] + ":" + color + ":" + str(ev)
    else:
        return (epd_i[0 : len(epd_i) - 6] + ":" + color + ":" + str(wdl), epd_i[0 : len(epd_i) - 6] + ":" + color + ":" + str(ev))    

In [41]:
# which can be one of ["wdl", "eval", "both"]
def generate_network(starting_fen, stockfish, max_depth, filen, which="wdl", decisive_threshold=2.0):
    if which != "both": 
        filename = f"{filen}_{which}_{max_depth}"
        file = open(f"{filename}.adj", "w")
        S = dict()
        starting_color = starting_fen.split(" ")[1]
        board = chess.Board(starting_fen)
        stack = [(board.fen(), 0)]
        S[get_node_name(board, starting_color, stockfish, decisive_threshold, which)] = 0

        while stack:
            print(f"{len(stack)}, {len(S)}       ", end="\r")
            fen_i, current_depth = stack.pop()
            board = chess.Board(fen_i)
            i = get_node_name(board, starting_color, stockfish, decisive_threshold, which)
            
            if board.is_game_over():
                continue
                
            if current_depth >= max_depth:
                continue
                
            legal_moves = board.legal_moves
            for move in legal_moves:
                board.push(move)
                fen_j = board.fen()
                j = get_node_name(board, starting_color, stockfish, decisive_threshold, which)

                next_depth = current_depth + 1
                if j not in S or next_depth < S[j]:
                    S[j] = next_depth
                    stack.append((fen_j, next_depth))
                file.write(f"{i} {j}\n")
                
                board.pop()

        file.close()
    else:
        filename1 = f"{filen}_wdl_{max_depth}"
        file1 = open(f"{filename1}.adj", "w")
        filename2 = f"{filen}_eval_{max_depth}"
        file2 = open(f"{filename2}.adj", "w")
        for file, which in zip([file1, file2], ["wdl", "eval"]):
            S = dict()
            starting_color = starting_fen.split(" ")[1]
            board = chess.Board(starting_fen)
            stack = [(board.fen(), 0)]
            S[get_node_name(board, starting_color, stockfish, decisive_threshold, which)] = 0

            while stack:
                print(f"{len(stack)}, {len(S)}       ", end="\r")
                fen_i, current_depth = stack.pop()
                board = chess.Board(fen_i)
                i = get_node_name(board, starting_color, stockfish, decisive_threshold, which)
                
                if board.is_game_over():
                    continue
                    
                if current_depth >= max_depth:
                    continue
                    
                legal_moves = board.legal_moves
                for move in legal_moves:
                    board.push(move)
                    fen_j = board.fen()
                    j = get_node_name(board, starting_color, stockfish, decisive_threshold, which)

                    next_depth = current_depth + 1
                    if j not in S or next_depth < S[j]:
                        S[j] = next_depth
                        stack.append((fen_j, next_depth))
                    file.write(f"{i} {j}\n")
                    
                    board.pop()

            file.close()

In [42]:
starting_fen = "8/3k4/7N/4K2p/7N/8/8/8 w - - 0 1"                         # KNNvKP tricky endgame
starting_fen = "4k3/8/1p1p3p/pPpPp1pP/P1P1PpP1/5P2/8/1K2RR2 w - - 0 1"   # fortress 1
#starting_fen = "8/8/8/1k3p2/p1p1pPp1/PpPpP1Pp/1P1P3P/QNK2NRR w - - 0 1"  # fortress 2

stockfish = Stockfish(path="C:/stockfish/stockfish-windows-x86-64-avx2")
generate_network(starting_fen, stockfish, 5, "fortress1", which="both")